# 📱 Classification de Spams SMS
**Auteur : Djiby KEBE**  
**Objectif :** Entraîner un modèle de Machine Learning pour détecter automatiquement les spams dans des SMS.  
**Dataset :** SMS Spam Collection (UCI Machine Learning Repository)  
**Algorithme :** Naive Bayes (Multinomial) + TF-IDF

---

## 1. 📦 Importation des bibliothèques

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print('✅ Bibliothèques importées avec succès')

ModuleNotFoundError: No module named 'sklearn'

## 2. 📂 Chargement des données

Le dataset est téléchargeable ici :  
👉 https://archive.ics.uci.edu/ml/datasets/SMS+Spam+Collection

Ou directement via la commande ci-dessous.

In [ ]:
# Téléchargement automatique du dataset
import urllib.request
import zipfile
import os

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
zip_path = 'smsspamcollection.zip'

if not os.path.exists('SMSSpamCollection'):
    print('Téléchargement du dataset...')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')
    print('✅ Dataset téléchargé et extrait')
else:
    print('✅ Dataset déjà présent')

# Chargement dans un DataFrame
df = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])
print(f'\nDataset chargé : {df.shape[0]} SMS')
df.head(10)

## 3. 🔍 Exploration des données (EDA)

In [ ]:
# Distribution des classes
print('=== Distribution des classes ===')
print(df['label'].value_counts())
print(f"\nPourcentage de spam : {df['label'].value_counts(normalize=True)['spam']*100:.1f}%")
print(f"Pourcentage de ham  : {df['label'].value_counts(normalize=True)['ham']*100:.1f}%")

In [ ]:
# Visualisation de la distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Graphique 1 : Distribution des classes
colors = ['#2ecc71', '#e74c3c']
df['label'].value_counts().plot(
    kind='bar', ax=axes[0], color=colors, edgecolor='black', rot=0
)
axes[0].set_title('Distribution des classes', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Nombre de SMS')
axes[0].bar_label(axes[0].containers[0])

# Graphique 2 : Longueur des messages
df['length'] = df['message'].apply(len)
df.groupby('label')['length'].plot(
    kind='hist', bins=50, alpha=0.6, ax=axes[1], legend=True
)
axes[1].set_title('Longueur des messages par classe', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Nombre de caractères')
axes[1].set_ylabel('Fréquence')

plt.tight_layout()
plt.savefig('distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Observation : les spams sont généralement plus longs que les ham.')

## 4. 🧹 Préparation des données

In [ ]:
import re
import string

def clean_text(text):
    """
    Nettoyage du texte :
    - Conversion en minuscules
    - Suppression de la ponctuation
    - Suppression des espaces superflus
    """
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Application du nettoyage
df['clean_message'] = df['message'].apply(clean_text)

# Encodage des labels : spam = 1, ham = 0
df['label_num'] = df['label'].map({'spam': 1, 'ham': 0})

print('Avant nettoyage :', df['message'][0])
print('Après nettoyage :', df['clean_message'][0])
print('\n✅ Nettoyage terminé')

## 5. ✂️ Séparation Train / Test

In [ ]:
X = df['clean_message']
y = df['label_num']

# 80% entraînement, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Données d\'entraînement : {len(X_train)} SMS')
print(f'Données de test        : {len(X_test)} SMS')
print(f'\nRépartition train - spam: {y_train.sum()} | ham: {(y_train==0).sum()}')
print(f'Répartition test  - spam: {y_test.sum()} | ham: {(y_test==0).sum()}')

## 6. 🔢 Vectorisation TF-IDF

**TF-IDF** (Term Frequency - Inverse Document Frequency) transforme chaque message en vecteur numérique.  
Un mot fréquent dans un message mais rare dans l'ensemble du dataset reçoit un score élevé — ce qui aide à identifier les mots caractéristiques des spams.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,   # On garde les 5000 mots les plus importants
    stop_words='english' # On ignore les mots vides (the, is, at...)
)

# Entraînement du vectoriseur sur les données d'entraînement UNIQUEMENT
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f'Taille du vocabulaire : {len(vectorizer.vocabulary_)} mots')
print(f'Forme de la matrice d\'entraînement : {X_train_tfidf.shape}')
print(f'Forme de la matrice de test        : {X_test_tfidf.shape}')

## 7. 🤖 Entraînement du modèle Naive Bayes

**Naive Bayes Multinomial** est particulièrement adapté à la classification de texte.  
Il calcule la probabilité qu'un message soit un spam en se basant sur les mots qu'il contient.

In [ ]:
# Entraînement
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

print('✅ Modèle entraîné avec succès')

## 8. 📊 Évaluation du modèle

In [ ]:
# Prédictions
y_pred = model.predict(X_test_tfidf)

# Métriques
accuracy = accuracy_score(y_test, y_pred)
print(f'=== Résultats ===')
print(f'Accuracy  : {accuracy * 100:.2f}%')
print(f'\n{classification_report(y_test, y_pred, target_names=["ham", "spam"])}')

In [ ]:
# Matrice de confusion
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['ham', 'spam'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matrice de Confusion', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\nVrais Négatifs  (ham correct)   : {tn}')
print(f'Faux Positifs   (ham → spam)    : {fp}')
print(f'Faux Négatifs   (spam → ham)    : {fn}')
print(f'Vrais Positifs  (spam correct)  : {tp}')

## 9. 🔑 Analyse des mots les plus discriminants

In [ ]:
# Mots les plus associés au spam et au ham
feature_names = vectorizer.get_feature_names_out()
log_probs = model.feature_log_prob_  # shape: (2, n_features)

# Top 15 mots spam
top_spam_idx = log_probs[1].argsort()[-15:][::-1]
top_ham_idx  = log_probs[0].argsort()[-15:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(feature_names[top_spam_idx][::-1], 
             log_probs[1][top_spam_idx][::-1], color='#e74c3c')
axes[0].set_title('Top 15 mots — SPAM', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Log-probabilité')

axes[1].barh(feature_names[top_ham_idx][::-1], 
             log_probs[0][top_ham_idx][::-1], color='#2ecc71')
axes[1].set_title('Top 15 mots — HAM', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Log-probabilité')

plt.tight_layout()
plt.savefig('top_words.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. 🧪 Test sur de nouveaux messages

In [ ]:
def predict_spam(message):
    """
    Prédit si un message est un spam ou non.
    Retourne le label et la probabilité associée.
    """
    cleaned = clean_text(message)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    proba = model.predict_proba(vectorized)[0]
    label = '🚨 SPAM' if prediction == 1 else '✅ HAM'
    confidence = proba[prediction] * 100
    print(f'Message   : "{message}"')
    print(f'Résultat  : {label} (confiance : {confidence:.1f}%)')
    print()

# Tests
predict_spam("Congratulations! You've won a FREE iPhone. Click here to claim now!")
predict_spam("Hey, are we still on for dinner tonight?")
predict_spam("URGENT: Your account has been compromised. Call us immediately at 0800-FREE")
predict_spam("Can you send me the notes from today's lecture?")
predict_spam("Win £1000 cash! Text WIN to 80080 now!")

## 11. 📝 Conclusion

### Résultats obtenus
- **Accuracy ~97%** sur les données de test
- Modèle léger, rapide à entraîner et facilement interprétable
- Les mots les plus discriminants pour le spam : *free, win, call, prize, claim*

### Ce que j'ai appris
- **TF-IDF** : transformer du texte brut en représentation numérique utilisable par un modèle ML
- **Naive Bayes** : un classifieur probabiliste simple mais très efficace sur les données textuelles
- **Évaluation** : lire une matrice de confusion et interpréter precision/recall/F1-score
- L'importance de ne pas appliquer `fit_transform` sur les données de test (fuite de données)

### Pistes d'amélioration
- Tester d'autres algorithmes : **SVM**, **Random Forest**, **Logistic Regression**
- Appliquer une **lemmatisation** pour normaliser les mots (running → run)
- Gérer le déséquilibre des classes avec **SMOTE** ou une pondération
- Déployer le modèle via une **API Flask** pour une utilisation en temps réel

---
*Projet réalisé dans le cadre d'une démarche d'autoformation en Machine Learning — Djiby KEBE, 2026*